### This is the paramter optimization technique for deccereasing the token consumption. where we are optimiziing the parameters fot different use cases such that the token consumption becomes less that will in turn reduce the cost of the api token consumption

In [1]:
import os
import json
import getpass
from typing import Dict, Any
from groq import Groq

In [2]:
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [39]:
INFERENCE_PROFILES = {
    "factual_qa": {
        "temperature": 0.15,
        "top_p": 0.8,
        "max_completion_tokens": 400,
        "system_prompt": (
            "You are a precise factual assistant. "
            "Answer directly. Avoid unnecessary explanation."
        )
    },

    "code_generation": {
        "temperature": 0.15,
        "top_p": 0.9,
        "max_completion_tokens": 1200,
        "system_prompt": (
            "You are a senior software engineer. "
            "Return clean, executable code with minimal explanation."
        )
    },

    "debugging": {
        "temperature": 0.1,
        "top_p": 0.85,
        "max_completion_tokens": 1000,
        "system_prompt": (
            "You are a debugging expert. "
            "Find the root cause first, then provide the smallest correct fix."
        )
    },

    "summarization": {
        "temperature": 0.1,
        "top_p": 0.8,
        "max_completion_tokens": 500,
        "system_prompt": (
            "You summarize content with high compression. "
            "Preserve only important facts, decisions, numbers, and actions."
        )
    },

    "json_extraction": {
        "temperature": 0.0,
        "top_p": 0.7,
        "max_completion_tokens": 700,
        "system_prompt": (
            "Extract structured information. "
            "Return only valid JSON. No markdown. No explanation."
        )
    },

    "creative": {
        "temperature": 0.8,
        "top_p": 0.95,
        "max_completion_tokens": 1000,
        "system_prompt": (
            "You are a creative assistant. "
            "Generate engaging, original, expressive content."
        )
    }
}

In [40]:
def detect_use_case(query: str):
    q = query.lower()

    # Highest Priority
    if any(x in q for x in [
        "error",
        "exception",
        "bug",
        "fix",
        "issue",
        "traceback",
        "stack trace",
        "failing",
        "not working"
    ]):
        return "debugging"

    if any(x in q for x in [
        "json",
        "extract",
        "schema",
        "fields",
        "parse"
    ]):
        return "json_extraction"

    if any(x in q for x in [
        "summarize",
        "summary",
        "shorten",
        "compress"
    ]):
        return "summarization"

    if any(x in q for x in [
        "story",
        "poem",
        "caption",
        "creative"
    ]):
        return "creative"

    if any(x in q for x in [
        "code",
        "function",
        "script",
        "notebook",
        "api"
    ]):
        return "code_generation"

    return "factual_qa"

In [41]:
def ask_groq_controlled(
    query: str,
    use_case: str = None,
    model: str = "llama-3.3-70b-versatile",
    seed: int = 42
) -> Dict[str, Any]:

    if use_case is None:
        use_case = detect_use_case(query)

    profile = INFERENCE_PROFILES[use_case]

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": profile["system_prompt"]
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=profile["temperature"],
        top_p=profile["top_p"],
        max_completion_tokens=profile["max_completion_tokens"],
        seed=seed
    )

    answer = response.choices[0].message.content
    usage = response.usage

    result = {
        "use_case": use_case,
        "model": model,
        "temperature": profile["temperature"],
        "top_p": profile["top_p"],
        "max_completion_tokens": profile["max_completion_tokens"],
        "answer": answer,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens
    }

    return result

In [42]:
def print_result(result: Dict[str, Any]):
    print("========== INFERENCE PROFILE ==========")
    print(f"Use Case              : {result['use_case']}")
    print(f"Model                 : {result['model']}")
    print(f"Temperature           : {result['temperature']}")
    print(f"Top-p                 : {result['top_p']}")
    print(f"Max Completion Tokens : {result['max_completion_tokens']}")

    print("\n========== RESPONSE ==========\n")
    print(result["answer"])

    print("\n========== TOKEN USAGE ==========")
    print(f"Input Tokens  : {result['input_tokens']}")
    print(f"Output Tokens : {result['output_tokens']}")
    print(f"Total Tokens  : {result['total_tokens']}")

### Use case - 1 Factual QA

In [35]:
query = "Explain Retrieval Augmented Generation in simple terms with one real-world example."

result = ask_groq_controlled(query)
print_result(result)

========== INFERENCE PROFILE ==========
Use Case              : factual_qa
Model                 : llama-3.3-70b-versatile
Temperature           : 0.2
Top-p                 : 0.8
Max Completion Tokens : 400

========== RESPONSE ==========

Retrieval Augmented Generation: A system that combines a knowledge retrieval step with a generation step to produce more accurate and informative outputs.

Example: A chatbot uses Retrieval Augmented Generation to answer a user's question "What are the top tourist attractions in Paris?" by first retrieving relevant information from a database (e.g., the Eiffel Tower, Louvre Museum) and then generating a response that incorporates this information, such as "The top tourist attractions in Paris include the Eiffel Tower and the Louvre Museum, which are both must-visit destinations."

========== TOKEN USAGE ==========
Input Tokens  : 65
Output Tokens : 116
Total Tokens  : 181


### Use case - 2 Code generation

In [36]:
query = "Write a Python function to call Groq API and track token usage."

result = ask_groq_controlled(query)
print_result(result)

========== INFERENCE PROFILE ==========
Use Case              : code_generation
Model                 : llama-3.3-70b-versatile
Temperature           : 0.15
Top-p                 : 0.9
Max Completion Tokens : 1200

========== RESPONSE ==========

```python
import requests
import json

class GroqAPI:
    def __init__(self, api_key, api_url="https://api.groq.com"):
        """
        Initialize the Groq API client.

        Args:
        - api_key (str): The API key for authentication.
        - api_url (str): The base URL of the Groq API. Defaults to "https://api.groq.com".
        """
        self.api_key = api_key
        self.api_url = api_url
        self.token_usage = 0

    def call_api(self, endpoint, method="GET", data=None):
        """
        Make a call to the Groq API.

        Args:
        - endpoint (str): The endpoint to call.
        - method (str): The HTTP method to use. Defaults to "GET".
        - data (dict): The data to send with the request. Defaults to None.



### Use case - 3 Summarization

In [37]:
query = """Summarize this in 5 bullet points:

ProdSync is an AI-powered SaaS platform for talent discovery, candidate evaluation,
and hiring intelligence. It helps job seekers improve employability using ATS resume
analysis, skill-gap detection, AI role recommendation, JD-resume matching, GitHub
project audit, readiness scoring, and AI mock interviews. For recruiters, it provides
semantic candidate search, AI shortlisting, automated first-round interview analysis,
and hireability reports. The platform targets Tier-2 and Tier-3 college students,
placement cells, and hiring teams in India."""

result = ask_groq_controlled(query)
print_result(result)

========== INFERENCE PROFILE ==========
Use Case              : summarization
Model                 : llama-3.3-70b-versatile
Temperature           : 0.1
Top-p                 : 0.8
Max Completion Tokens : 500

========== RESPONSE ==========

Here are 5 key points about ProdSync:
* ProdSync is an AI-powered SaaS platform for talent discovery and hiring intelligence.
* It helps job seekers with resume analysis, skill-gap detection, and AI mock interviews.
* The platform assists recruiters with semantic candidate search, AI shortlisting, and automated interview analysis.
* ProdSync provides features like JD-resume matching, GitHub project audit, and hireability reports.
* The platform targets Tier-2 and Tier-3 college students, placement cells, and hiring teams in India.

========== TOKEN USAGE ==========
Input Tokens  : 170
Output Tokens : 108
Total Tokens  : 278


### Use case - 4 Debugging

In [44]:
query = """I am getting this error:

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Code:
age = 25
message = "My age is " + age
print(message)

Find the bug and give the smallest correct fix."""

result = ask_groq_controlled(query)
print_result(result)

========== INFERENCE PROFILE ==========
Use Case              : debugging
Model                 : llama-3.3-70b-versatile
Temperature           : 0.1
Top-p                 : 0.85
Max Completion Tokens : 1000

========== RESPONSE ==========

**Root Cause:** 
The error occurs because you're trying to concatenate a string with an integer using the `+` operator, which is not allowed in Python.

**Smallest Correct Fix:**
You can fix this by converting the integer to a string using the `str()` function:

```python
age = 25
message = "My age is " + str(age)
print(message)
```

Alternatively, you can use string formatting or f-strings for a more modern and readable approach:

```python
age = 25
message = f"My age is {age}"
print(message)
```

========== TOKEN USAGE ==========
Input Tokens  : 107
Output Tokens : 123
Total Tokens  : 230


### Use cae -5 JSON Extraction

In [48]:
query = """Extract the following information as JSON:

Candidate Name: Soumyajit Bera
Current Role: AI Engineer
Company: IBM
Experience: 2 years 9 months
Skills: Python, FastAPI, LangChain, Milvus, SQL, Machine Learning
Expected CTC: 25 LPA
Preferred Location: Kolkata

Return fields:
name, current_role, company, experience, skills, expected_ctc, preferred_location"""

result = ask_groq_controlled(query)
print_result(result)

========== INFERENCE PROFILE ==========
Use Case              : json_extraction
Model                 : llama-3.3-70b-versatile
Temperature           : 0.0
Top-p                 : 0.7
Max Completion Tokens : 700

========== RESPONSE ==========

{
  "name": "Soumyajit Bera",
  "current_role": "AI Engineer",
  "company": "IBM",
  "experience": "2 years 9 months",
  "skills": ["Python", "FastAPI", "LangChain", "Milvus", "SQL", "Machine Learning"],
  "expected_ctc": "25 LPA",
  "preferred_location": "Kolkata"
}

========== TOKEN USAGE ==========
Input Tokens  : 139
Output Tokens : 89
Total Tokens  : 228


### use case - 6 Creative

In [50]:
query = """
Write a powerful LinkedIn post announcing ProdSync as an AI-powered employability
and hiring intelligence platform for Tier-2 and Tier-3 college students in India.
Keep it founder-style, confident, and inspiring.
"""
result = ask_groq_controlled(query)
print_result(result)

========== INFERENCE PROFILE ==========
Use Case              : factual_qa
Model                 : llama-3.3-70b-versatile
Temperature           : 0.15
Top-p                 : 0.8
Max Completion Tokens : 400

========== RESPONSE ==========

"Revolutionizing Employability in India

I'm thrilled to introduce ProdSync, the game-changing AI-powered employability and hiring intelligence platform, specifically designed for Tier-2 and Tier-3 college students in India.

Our mission is to bridge the gap between talent and opportunity, empowering millions of students to unlock their full potential. With ProdSync, we're committed to transforming the hiring landscape, making it more inclusive, efficient, and effective.

By leveraging cutting-edge AI technology, we provide personalized skill development, job matching, and career guidance, ensuring that every student has an equal chance to succeed. Our platform is dedicated to helping colleges and universities enhance their placement outcomes, while